In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
from tueplots import bundles, cycler, figsizes
from tueplots.constants.color import palettes
import tueplots.constants.color.palettes as tuepalt

plt.rcParams.update(bundles.icml2022())
plt.rcParams.update(cycler.cycler(color=palettes.tue_plot))
plt.rcParams.update({"figure.dpi": 350})

In [ ]:
os.makedirs("fig", exist_ok=True)

In [ ]:
path = "../../data/newspaper_collection_evaluation_results_20_12_2025.csv"
np_coll_df = pd.read_csv(path, index_col=False)

In [ ]:
# map emotions to integers
available_emotions = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "sad",
    "surprise",
    "neutral",
]
emotions_id_map = {e: i for i, e in enumerate(available_emotions)}
id_emotions_map = {i: e for i, e in enumerate(available_emotions)}

In [ ]:
np_coll_df = np_coll_df.query("party != 'transnational'")

In [ ]:
# some preprocessing
np_coll_df["date"] = pd.to_datetime(np_coll_df["date"])
np_coll_df["dominant_emotion_id"] = np_coll_df["dominant_emotion"].map(emotions_id_map)
emotions_str = ['angry','fear', 'happy', 'sad', 'surprise', 'neutral']

### Available Attributes:
- **personal attributes**
    - name, 
    - surname, 
    - fullname, 
    - birth, 
    - gender, 
- **additional info**
    - article, 
    - party, 
    - newspaper, 
- **evaluation metadata**
    - confidence, 
    - distance, 
    - date, 
- **emotions**:
    - dominant_emotion, 
    - angry, 
    - disgust, 
    - fear, 
    - happy, 
    - sad, 
    - surprise, 
    - neutral, 

In [ ]:
np_coll_df["newspaper"].unique()

In [ ]:
newspaper_alignment = {
    "cmpct": "Right-wing extremist / New Right",
    "stern": "Social-liberal / Centrist",
    "spgl": "Social-liberal / Center-left",
    "sz": "Liberal-left / Center-left",
    "taz": "Green-left / Alternative",
    "frtg": "Left-liberal / Intellectual",
    "nd": "Democratic socialist / Left-wing"
}

In [ ]:
newspaper_name_mapping = {
    "sz": "sz",
    "stern": "stern",
    "freitag": "frtg",
    "taz": "taz",
    "compact": "cmpct",
    "spiegel": "spgl",
    "nd": "nd"
}
np_coll_df["newspaper"] = np_coll_df["newspaper"].map(newspaper_name_mapping)

In [ ]:
party_name_mapping = {
    "afd": "afd",
    "transnational": "trnsntl",
    "gruenen": "gruene",
    "spd": "spd",
    "linke": "linke",
    "union": "union"
}

np_coll_df["party"] = np_coll_df["party"].map(party_name_mapping)

## Chose a confidence to work with

In [ ]:
min_confidence = 75
cut_off_confidence_df = np_coll_df.query("confidence > @min_confidence")
confidence_mean = cut_off_confidence_df["confidence"].mean()
print("*================================================*")
print(f"| {len(cut_off_confidence_df)}/{len(np_coll_df)} Samples remain with confidence > {min_confidence} |")
print(f"|       Resulting confidence mean: {confidence_mean:.2f}         |")
print("*================================================*")
np_coll_df = cut_off_confidence_df

## Drop Rows that make only a small proportion 

In [ ]:
np_coll_df = np_coll_df.query("party != 'fdp'")
np_coll_df = np_coll_df.drop(columns=["disgust"])
np_coll_df.head(1)

In [ ]:
def barplot_factory(data, x, y, hue, ax, alpha=1, width=1, mean_attr=None):
    if mean_attr:
        overall_mean = np_coll_df[mean_attr].mean()

    sns.barplot(data=data, x=x, y=y, hue=hue, alpha=alpha, width=width, ax=ax, dodge=False)
    if mean_attr: ax.axhline(overall_mean, color='darkred', linestyle='--', linewidth=1)
    
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, fmt='%.2f', label_type='edge', padding=-15, color='white', fontweight='bold')

In [ ]:
fig, ax = plt.subplots(1, 2)
barplot_factory(data=np_coll_df, x="party", y="confidence", hue="party", alpha=0.9, width=0.7, ax=ax[0], mean_attr="confidence")
barplot_factory(data=np_coll_df, x="newspaper", y="confidence", hue="newspaper", alpha=0.9, width=0.7, ax=ax[1], mean_attr="confidence")
plt.savefig(f"fig/confidence_distr_party_newspaper_cutoff_{min_confidence}_mean_{confidence_mean}.pdf")
plt.show()

## How is the data distributed?

In [ ]:
os.makedirs("fig/distr", exist_ok=True)
# party
fig, ax = plt.subplots(2, 2, figsize=(8, 3))
fig.suptitle("Distribution of Samples by Different Categories")
sns.countplot(data=np_coll_df, x="party", hue="party", ax=ax[0, 0], order=np_coll_df["party"].value_counts().index)
sns.countplot(data=np_coll_df, x="newspaper", hue="newspaper", ax=ax[0, 1], order=np_coll_df["newspaper"].value_counts().index)
sns.countplot(data=np_coll_df, x="gender", hue="gender", ax=ax[1, 0], order=np_coll_df["gender"].value_counts().index)
sns.countplot(data=np_coll_df, x="dominant_emotion", hue="dominant_emotion", ax=ax[1, 1], order=np_coll_df["dominant_emotion"].value_counts().index)
plt.savefig("fig/distr/sample_distribution_party_gender_newspaper_emotion.pdf")
plt.show()

In [ ]:
np_coll_df["party"]

In [ ]:
# party, emotion, newspaper
grouped_emotions_df = np_coll_df.groupby(by=["newspaper", "party", "date"])[emotions_str].mean()
grouped_emotions_df

In [ ]:
import matplotlib.dates as mdates
plot_cell = False
if plot_cell:
    for e in emotions_str:
        data = grouped_emotions_df.loc[("compact", "afd"), [e]].reset_index()
        df_long = data.melt(id_vars="date", var_name="emotion", value_name="score")
        fig, ax = plt.subplots(figsize=(6, 2))
        sns.lineplot(df_long, x="date", y="score", hue="emotion", ax=ax)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        plt.xticks(rotation=25)
        plt.show()

In [ ]:
rows, cols = 2, 3
fig, ax = plt.subplots(rows, cols, figsize=(5, 3))
for i in range(rows):
    for j in range(cols):
        idx = (i*cols) + j
        c_emotion = emotions_str[idx]
        agg_party_newspaper = np_coll_df.groupby(["party", "newspaper"], as_index=False)[c_emotion].mean()
        ax[i, j].set_title(c_emotion)
        sns.scatterplot(
            data=agg_party_newspaper,
            x="newspaper",
            y="party",
            hue="newspaper",
            size=c_emotion,
            sizes=(1, 100),
            legend=False,
            alpha=0.8,
            marker="8",
            ax=ax[i, j],
            edgecolor="black",
            linewidth=0.5,
        )
        ax[i, j].set_xlabel("")
        ax[i, j].set_ylabel("")
plt.tight_layout()
plt.savefig("fig/mean_emotions_by_party_newspaper.pdf", dpi=300)

In [ ]:
plot_cell = True
if plot_cell:
    for i in range(rows):
        for j in range(cols):
            fig, ax = plt.subplots()
            idx = (i * cols) + j
            c_emotion = emotions_str[idx]
            agg_party_newspaper = np_coll_df.groupby(["party", "newspaper"], as_index=False)[c_emotion].mean()
            pivot_df = agg_party_newspaper.pivot(index="party", columns="newspaper", values=c_emotion)
            
            sns.heatmap(
                pivot_df, 
                ax=ax, 
                annot=True, 
                fmt=".2f", 
                cbar=False,
                annot_kws={"fontsize": 8}
            )
            ax.set_title(c_emotion)
            ax.set_xlabel(None)
            ax.set_ylabel(None)
            # plt.savefig("fig/heatmap_emotion_by_party_newspaper.pdf")

In [ ]:
emotions_diff_order = ["angry", "happy", "sad", "surprise", "neutral", "fear"]
rows, cols = 2, 3
plt.rcParams.update(bundles.icml2024(column="full", nrows=rows, ncols=cols))
fig, ax = plt.subplots(rows, cols)
for i in range(rows):
    for j in range(cols):
        idx = (i * cols) + j
        c_emotion = emotions_diff_order[idx]
        agg_party_newspaper = np_coll_df.groupby(["party", "newspaper"], as_index=False)[c_emotion].mean()
        pivot_df = agg_party_newspaper.pivot(index="newspaper", columns="party", values=c_emotion)
        
        sns.heatmap(
            pivot_df, 
            ax=ax[i, j], 
            annot=True, 
            fmt=".2f", 
            # cmap="YlGnBu", 
            cbar=False,
            square=False,
            annot_kws={"fontsize": 6}
        )
        ax[i, j].set_title(c_emotion)
        ax[i, j].set_xlabel(None)
        ax[i, j].set_ylabel(None)
plt.savefig("fig/heatmap_emotion_by_party_newspaper.pdf")

In [ ]:
# "circle/scatter plot"
plot_cell = False
if plot_cell: 
    rows, cols = 2, 3
    fig, ax = plt.subplots(rows, cols, figsize=(12, 7))

    for i in range(rows):
        for j in range(cols):
            idx = (i * cols) + j
            c_emotion = emotions_str[idx]
            agg_party_newspaper = np_coll_df.groupby(["party", "newspaper"], as_index=False)[c_emotion].mean()
            pivot_df = agg_party_newspaper.pivot(index="party", columns="newspaper", values=c_emotion)
            ax[i, j].set_title(c_emotion)
            sns.scatterplot(
                data=agg_party_newspaper,
                x="newspaper",
                y="party",
                hue="newspaper",
                size=c_emotion,
                sizes=(100, 1000),
                legend=False,
                alpha=0.8,
                marker="o",
                ax=ax[i, j],
                edgecolor="black",
                linewidth=1.5
            )
            ax[i, j].set_xlabel("")
            ax[i, j].set_ylabel("")
            ax[i, j].margins(x=0.1, y=0.1)

    plt.tight_layout()

## Do the same again, but with dominant emotions

In [ ]:
dominant_emotion_df = np_coll_df[
    ["name", "surname", "party", "newspaper", "dominant_emotion", "dominant_emotion_id", "date"] + emotions_str].copy()
dominant_emotion_df = dominant_emotion_df.reset_index(drop=True)

In [ ]:
dominant_emotion_df[available_emotions] = 0
for i in range(len(dominant_emotion_df)):
    row = dominant_emotion_df.iloc[i]
    row_dominant_emotion = row["dominant_emotion"]
    row_dominant_emotion_id = emotions_id_map[row_dominant_emotion]
    dominant_emotion_df.loc[i, row_dominant_emotion] = 1

In [ ]:
entries = 0
for e in emotions_str:
    agg_dominant_emotions_df = dominant_emotion_df.groupby(["newspaper", "party"], as_index=False)[e].sum()
    entries += agg_dominant_emotions_df[e].sum()

In [ ]:
def capitalize_first_letter(string):
    first_letter = string[0].upper()
    res = string[1:]
    result = first_letter + res
    return result

s = "test"
capitalize_first_letter("test")

In [ ]:
plot = False
if plot:
    rows, cols = 2, 3
    fig, ax = plt.subplots(rows, cols)
    for i in range(rows):
        for j in range(cols):
            fig_, ax_save = plt.subplots()
            idx = (i * cols) + j
            c_emotion = emotions_str[idx]
            agg_dominant_emotions_df = dominant_emotion_df.groupby(["newspaper", "party"], as_index=False)[c_emotion].mean()
            pivot_df = agg_dominant_emotions_df.pivot_table(
                index="newspaper", 
                columns="party", 
                values=c_emotion, 
                aggfunc='mean'
            )
            
            sns.heatmap(
                pivot_df, 
                ax=ax[i, j], 
                annot=True, 
                fmt=".2f", 
                # cmap="YlGnBu", 
                cbar=False,
                square=False,
                annot_kws={"fontsize": 6}
            )
            ax[i, j].set_title(capitalize_first_letter(c_emotion), fontsize=6)
            ax[i, j].set_xlabel(None)
            ax[i, j].set_ylabel(None)
    plt.savefig("fig/heatmap_dominant_emotion_by_party_newspaper.pdf")

In [ ]:
emotions_str

In [ ]:
emotions_diff_order = ["angry", "happy", "sad", "surprise", "neutral", "fear"]
rows, cols = 2, 3
fig, ax = plt.subplots(rows, cols)
for i in range(rows):
    for j in range(cols):
        idx = (i * cols) + j
        c_emotion = emotions_diff_order[idx]
        agg_dominant_emotions_df = dominant_emotion_df.groupby(["newspaper", "party"], as_index=False)[c_emotion].mean()
        pivot_df = agg_dominant_emotions_df.pivot_table(
            index="newspaper", 
            columns="party", 
            values=c_emotion, 
            aggfunc='mean'
        )
        
        sns.heatmap(
            pivot_df, 
            ax=ax[i, j], 
            annot=True, 
            fmt=".2f", 
            # cmap="YlGnBu", 
            cbar=False,
            square=False,
            annot_kws={"fontsize": 6}
        )
        ax[i, j].set_title(capitalize_first_letter(c_emotion), fontsize=6)
        ax[i, j].set_xlabel(None)
        ax[i, j].set_ylabel(None)
plt.savefig("fig/heatmap_dominant_emotion_by_party_newspaper.pdf")

### Take Closer Look at Parties, Newspaper, and Happy, Sad, Angry 

In [ ]:
y_label_map = {"angry": "anger", "neutral": "neutrality", "happy":"happiness", "sad": "sadness"}

def make_emotion_party_newspaper_plots(df, parties: list, emotion: str, newspapers: list, ax, type="bar"): 
    df_melted = df.melt(
        id_vars=["party", "newspaper"], 
        value_vars=[emotion], 
        var_name="emotion", 
        value_name="value"
    )

    query_str = "emotion == @emotion and party in @parties and newspaper in @newspapers"
    plot_df = df_melted.query(query_str)
    
    emotion_mean = plot_df["value"].mean()

    if type == "box":
        sns.boxplot(
            data=plot_df,
            x="party",
            y="value",
            hue="newspaper",
            ax=ax,
            fliersize=0.1,
            linewidth=0.5,
            notch=True,
            bootstrap=10000
        )
    elif type == "violin":
        sns.violinplot(
            data=plot_df,
            x="party",
            y="value",
            hue="newspaper",
            ax=ax,
            linewidth=0.5,
            cut=0,
            inner="quartile"
        )
    else:
        sns.barplot(
            data=plot_df,
            x="party",
            y="value",
            hue="newspaper",
            ax=ax,
            width=0.8,
            err_kws={'linewidth': 0.5},
            alpha=0.9,
            edgecolor="black",
            linewidth=0.5
        )

    ax.axhline(emotion_mean, color="crimson", linestyle="--", linewidth=0.5, zorder=10)
    ax.set_ylabel(f"proportion of {y_label_map[emotion]}")

In [ ]:
r, c = 1, 3
plt.rcParams.update(bundles.icml2024(column="full", nrows=r, ncols=c))
fig, ax = plt.subplots(r, c)
emotions = ["happy", "neutral", "angry"]
c_parties = np_coll_df["party"].unique().tolist()# ["afd", "gruene", "linke"]

for i, emo in enumerate(emotions):
    make_emotion_party_newspaper_plots(
        np_coll_df,
        parties=c_parties,
        emotion=emotions[i],
        newspapers=np_coll_df["newspaper"].unique().tolist(),
        ax=ax[i],
        type="box")


for ax_ in ax:
    ax_.get_legend().remove()

handles, labels = ax[0].get_legend_handles_labels()

legend = fig.legend(
    handles, 
    labels, 
    loc='upper center', 
    bbox_to_anchor=(0.5, 1.1), 
    ncol=7, 
    frameon=False
)
fig.savefig("fig/test.pdf", bbox_extra_artists=(legend,), bbox_inches="tight")
plt.show()

In [ ]:
r, c = 1, 3
plt.rcParams.update(bundles.icml2024(column="full", nrows=r, ncols=c))
fig, ax = plt.subplots(r, c)
emotions = ["neutral", "sad", "angry"]
c_parties = np_coll_df["party"].unique().tolist()# ["afd", "gruene", "linke"]

for i, emo in enumerate(emotions):
    make_emotion_party_newspaper_plots(
        np_coll_df.query("party != 'trnsntl'"),
        parties=c_parties,
        emotion=emotions[i],
        newspapers=np_coll_df["newspaper"].unique().tolist(),
        ax=ax[i],
        type="bar")

# Customize party labels on x-axis
party_label_map = {
    "afd": "AfD",
    "gruene": "Grünen",
    "spd": "SPD",
    "linke": "Linke",
    "union": "Union",
    "trnsntl": "Transnational"
}

# Apply custom party labels to x-axis tick labels
for ax_ in ax:
    ax_.get_legend().remove()
    # Get current x-tick labels and map them
    current_labels = [tick.get_text() for tick in ax_.get_xticklabels()]
    mapped_party_labels = [party_label_map.get(label, label) for label in current_labels]
    # Fix for matplotlib warning: set ticks before setting labels
    ax_.set_xticks(ax_.get_xticks())
    ax_.set_xticklabels(mapped_party_labels)
    ax_.set_xlabel("Party")

handles, labels = ax[0].get_legend_handles_labels()

# Customize legend labels (only newspapers appear in legend)
newspaper_label_map = {
    "cmpct": "Compact",
    "stern": "Stern",
    "spgl": "Der Spiegel",
    "sz": "Süddeutsche Zeitung",
    "taz": "Die Tageszeitung",
    "frtg": "Freitag",
    "nd": "Neues Deutschland"
}

# Apply custom labels to the labels list
mapped_labels = [newspaper_label_map.get(label, label) for label in labels]

fig.legend(
    handles, 
    mapped_labels, 
    loc='upper center', 
    bbox_to_anchor=(0.5, 1.1), 
    ncol=7, 
    frameon=False
)

plt.savefig("fig/newspaper_party_emotion_proportions.pdf", bbox_inches="tight")
plt.show()

In [ ]:
r, c = 1, 2
plt.rcParams.update(bundles.icml2024(column="full", nrows=r, ncols=c))
fig, ax = plt.subplots(r, c)
emotions = ["happy", "sad"]
c_parties = ["afd", "gruene", "linke"] # np_coll_df["party"].unique().tolist()

for i, emo in enumerate(emotions):
    make_emotion_party_newspaper_plots(
        np_coll_df,
        parties=c_parties,
        emotion=emotions[i],
        newspapers=np_coll_df["newspaper"].unique().tolist(),
        ax=ax[i],
        type="violin")

for ax_ in ax:
    ax_.get_legend().remove()

handles, labels = ax[0].get_legend_handles_labels()

fig.legend(
    handles, 
    labels, 
    loc='upper center', 
    bbox_to_anchor=(0.5, 1.1), 
    ncol=7, 
    frameon=False
)
plt.show()

## Checking for biases

In [ ]:
from scipy.stats import chi2_contingency
def test_pairwise_newspapers(df, newspaper_A, newspaper_B, target_emotion, target_party):
    subset = df.query("party == @target_party").copy()
    
    subset = subset[subset['newspaper'].isin([newspaper_A, newspaper_B])]
    subset['is_emotion'] = subset['dominant_emotion'] == target_emotion

    count_A_target = subset[(subset['newspaper'] == newspaper_A) & (subset['is_emotion'])].shape[0]
    count_A_other = subset[(subset['newspaper'] == newspaper_A) & (~subset['is_emotion'])].shape[0]
    
    count_B_target = subset[(subset['newspaper'] == newspaper_B) & (subset['is_emotion'])].shape[0]
    count_B_other = subset[(subset['newspaper'] == newspaper_B) & (~subset['is_emotion'])].shape[0]
    
    if (count_A_target + count_A_other == 0) or (count_B_target + count_B_other == 0):
        #print(f"cA {newspaper_A}:", count_A_target + count_A_other)
        #print(f"cB {newspaper_B}:", count_B_target + count_B_other)
        return None

    obs = [[count_A_target, count_A_other], [count_B_target, count_B_other]]
    chi2, p_val, _, _ = chi2_contingency(obs)
    
    prop_A = count_A_target / (count_A_target + count_A_other)
    prop_B = count_B_target / (count_B_target + count_B_other)
    
    return {
        "newspaper_A": newspaper_A,
        "newspaper_B": newspaper_B,
        "party": target_party,
        "emotion": target_emotion,
        "n_faces_A": count_A_target + count_A_other,
        "n_faces_B": count_B_target + count_B_other,
        "prop_A": round(prop_A, 3),
        "prop_B": round(prop_B, 3),
        "diff": round(prop_A - prop_B, 3),
        "p_value": round(p_val, 4),
        "significant": p_val < 0.05
    }

In [ ]:
all_newspapers = np_coll_df["newspaper"].unique().tolist()

npA = "spgl"
all_test_results = []
for npA in all_newspapers:
    
    for target_party in np_coll_df["party"].unique().tolist():

        for target_emotion in ["happy", "angry", "sad"]:
            test_results = []
            for np in all_newspapers:
                if np == npA: continue
                
                res = test_pairwise_newspapers(
                    np_coll_df,
                    newspaper_A=npA,
                    newspaper_B=np,
                    target_party=target_party,
                    target_emotion=target_emotion
                )
                
                if res:
                    test_results.append(res)
            
            if not test_results:
                continue

            test_results_df = pd.DataFrame(test_results)
            all_test_results.append(test_results_df) 
            mean_p = test_results_df['p_value'].mean()
            # print(f"{target_emotion}, {target_party}; [{npA}] vs. [all] => p-value: {mean_p}")

all_test_results_df = pd.concat(all_test_results)

In [ ]:
execute = True
if execute:
    all_test_results_df = all_test_results_df.reset_index(drop=True).reset_index()
execute = False

In [ ]:
r, c, = 1, 1
plt.rcParams.update(bundles.icml2024(column="full", nrows=r, ncols=c))
ax, fig = plt.subplots()
sns.kdeplot(
    data=all_test_results_df.query("emotion == 'happy'"),
    x="p_value",
    hue="newspaper_A",
    fill=True
)


In [ ]:
np